In [1]:
import pennylane as qml
from pennylane import numpy as np

In [2]:
# ============================================
# Device
# ============================================

dev = qml.device("default.qubit", wires=2)

In [3]:
# ============================================
# Cost Hamiltonian
# H = -1/2 I + 3/2 Z0 - 2 Z1 + Z0 Z1
# ============================================

coeffs = [-0.5, 1.5, -2.0, 1.0]

observables = [
    qml.Identity(0),
    qml.PauliZ(0),
    qml.PauliZ(1),
    qml.PauliZ(0) @ qml.PauliZ(1)
]

cost_h = qml.Hamiltonian(coeffs, observables)

# ============================================
# Mixer Hamiltonian
# ============================================

mixer_h = qml.Hamiltonian(
    [1.0, 1.0],
    [qml.PauliX(0), qml.PauliX(1)]
)

# ============================================
# QAOA Layer
# ============================================

def qaoa_layer(gamma, beta):

    qml.qaoa.cost_layer(gamma, cost_h)

    qml.qaoa.mixer_layer(beta, mixer_h)

# ============================================
# QAOA Circuit
# ============================================

p = 1

@qml.qnode(dev)
def circuit(params):

    # Initial superposition
    for wire in range(2):
        qml.Hadamard(wires=wire)

    # QAOA layers
    for i in range(p):
        qaoa_layer(params[0][i], params[1][i])

    return qml.expval(cost_h)

In [4]:
# ============================================
# Optimization
# ============================================

params = np.array([[0.1], [0.1]], requires_grad=True)

print(params)

optimizer = qml.AdamOptimizer(stepsize=0.1)

steps = 100

for i in range(steps):

    params = optimizer.step(circuit, params)

    if (i + 1) % 10 == 0:
        energy = circuit(params)
        print(f"Step {i+1:3d} | Energy = {energy:.6f}")


[[0.1]
 [0.1]]
Step  10 | Energy = -0.592174
Step  20 | Energy = -3.756747
Step  30 | Energy = -3.706536
Step  40 | Energy = -3.771536
Step  50 | Energy = -3.835376
Step  60 | Energy = -3.865969
Step  70 | Energy = -3.875819
Step  80 | Energy = -3.878528
Step  90 | Energy = -3.879281
Step 100 | Energy = -3.879420


In [5]:
# ============================================
# Final Results
# ============================================

final_energy = circuit(params)

print("\nOptimized Parameters:")
print(params)

print("\nGround-State Energy:")
print(final_energy)

# ============================================
# Probability Distribution
# ============================================

@qml.qnode(dev)
def probabilities(params):

    for wire in range(2):
        qml.Hadamard(wires=wire)

    for i in range(p):
        qaoa_layer(params[0][i], params[1][i])

    return qml.probs(wires=[0,1])

probs = probabilities(params)

print("\nProbabilities:")
print(probs)

states = ["00", "01", "10", "11"]

max_index = np.argmax(probs)

print(f"\nMost Probable Solution: {states[max_index]}")


Optimized Parameters:
[[ 0.35385891]
 [-0.73238649]]

Ground-State Energy:
-3.879420125513931

Probabilities:
[0.04746986 0.12267181 0.82577035 0.00408798]

Most Probable Solution: 10
